In [ ]:
from pathlib import Path
import json
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
from pycocotools import mask as mask_util

with open("../data/labeled/splits.json") as f:
    split_data = json.load(f)

splits = split_data.keys()


def binary_mask_to_rle(binary_mask):
    rle = mask_util.encode(np.asfortranarray(binary_mask))
    rle["counts"] = rle["counts"].decode(
        "utf-8"
    )  # Convert binary to string for JSON serialization
    return rle


for split in splits:
    folder = Path("../datasets/photobox_coco")
    images_folder = folder / "data"

    folder.mkdir(exist_ok=True, parents=True)
    images_folder.mkdir(exist_ok=True)

    labels = {
        "images": [],
        "categories": [{"id": 1, "name": "colony"}],
        "annotations": [],
    }
    annotation_id = 0

    for id, image_id in tqdm(
        enumerate(split_data[split]),
        desc=split,
        total=len(split_data[split]),
    ):
        file_name = f"{image_id}.jpg"

        image = Image.open(f"../data/labeled/images/{file_name}")
        image.save(images_folder / file_name)

        height = image.height
        width = image.width

        labels["images"].append(
            {
                "id": id,
                "file_name": file_name,
                "height": height,
                "width": width,
            }
        )

        mask_path = Path(f"../data/labeled/masks/{image_id}.png")
        mask = np.array(Image.open(mask_path))

        int_mask = mask.copy().astype(np.uint32)
        int_mask = (
            (int_mask[:, :, 2] << 16) + (int_mask[:, :, 1] << 8) + int_mask[:, :, 0]
        )

        unique_colors = np.unique(int_mask[int_mask != 0])
        unique_colors_count = len(unique_colors)

        metadata = [mask_path.name, unique_colors_count]

        for i, color in enumerate(unique_colors):
            binary_mask = (int_mask == color).astype(np.uint8)
            rle = binary_mask_to_rle(binary_mask)

            # Use pycocotools to get bounding box
            bbox = mask_util.toBbox(rle).tolist()

            area = float(mask_util.area(rle))

            labels["annotations"].append(
                {
                    "id": annotation_id,
                    "image_id": id,
                    "category_id": 1,
                    "area": area,
                    "bbox": bbox,
                    "segmentation": rle,
                    "iscrowd": 0,
                }
            )

            annotation_id += 1

    with open(folder / f"{split}.json", "w") as f:
        json.dump(labels, f)

In [ ]:
%pip install -q fiftyone

In [ ]:
import fiftyone as fo

dataset_train = fo.Dataset.from_dir(
    "../datasets/photobox_coco",
    labels_path="train.json",
    dataset_type=fo.types.COCODetectionDataset,
)

session = fo.launch_app(dataset_train, auto=False, port=5151)

print(f"Dataset launched at http://localhost:{session.server_port}")